# Temporal Analysis of Resistance Profiles

Three analyses:
- **A** — Which profiles are increasing or decreasing in frequency (trend ranking)
- **B** — How resistance burden zones evolve over time
- **C** — Direct comparison between first and last year (2018 vs 2024)

## Image of the Gephi network - Editing

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.lines as mlines
from matplotlib.colors import LinearSegmentedColormap
import numpy as np
from PIL import Image

# Cargar la imagen exportada de Gephi
img = Image.open('/Users/jimenamartinreina/Documents/PabloCatalan/REPOSITORY/data/binary_network_Gephi.png')

fig, ax = plt.subplots(figsize=(14, 8))
ax.imshow(img)
ax.axis('off')

# --- Anotaciones de zonas ---
ax.annotate('Susceptible\nzone', xy=(0.08, 0.48), xycoords='axes fraction',
            fontsize=10, fontweight='bold', color='#2d6a4f',
            ha='center',
            bbox=dict(boxstyle='round,pad=0.3', facecolor='white', edgecolor='#2d6a4f', alpha=0.8))

ax.annotate('Multidrug-resistant\nzone', xy=(0.88, 0.48), xycoords='axes fraction',
            fontsize=10, fontweight='bold', color='#1b4332',
            ha='center',
            bbox=dict(boxstyle='round,pad=0.3', facecolor='white', edgecolor='#1b4332', alpha=0.8))

# Flecha izquierda → derecha
ax.annotate('', xy=(0.82, 0.5), xycoords='axes fraction',
            xytext=(0.12, 0.5),
            arrowprops=dict(arrowstyle='->', color='gray', lw=1.5))
ax.text(0.48, 0.52, 'Increasing resistance burden',
        transform=ax.transAxes, fontsize=8, color='gray', ha='center')

# --- Leyenda de color ---
cmap = LinearSegmentedColormap.from_list('green_grad',
    ['#b7e4c7', '#2d6a4f'])  # ajusta a los colores exactos de tu red
sm = plt.cm.ScalarMappable(cmap=cmap, norm=plt.Normalize(vmin=0, vmax=19))
sm.set_array([])
cbar = fig.colorbar(sm, ax=ax, orientation='vertical',
                    fraction=0.02, pad=0.01, shrink=0.4)
cbar.set_label('Number of resistances', fontsize=9)
cbar.set_ticks([0, 5, 10, 15, 19])

# --- Leyenda de tamaño ---
size_legend_ax = fig.add_axes([0.01, 0.05, 0.12, 0.2])
size_legend_ax.axis('off')
for size, label in [(50, 'Low'), (150, 'Medium'), (300, 'High')]:
    size_legend_ax.scatter([], [], s=size, color='#52b788', alpha=0.7, label=label)
size_legend_ax.legend(title='Relative\nfrequency', fontsize=7, title_fontsize=8,
                      loc='center', framealpha=0.8)

plt.tight_layout()
plt.show()

## 0. Setup

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import matplotlib.colors as mcolors
from matplotlib.lines import Line2D

plt.rcParams.update({
    'figure.dpi': 150,
    'font.size': 10,
    'axes.spines.top': False,
    'axes.spines.right': False
})

# --- Load data ---
# The data can be found in `REPOSITORY/data/nodes_data.csv` and is generated from binary_network_3.gexf in Gephi
df = pd.read_csv('/Users/jimenamartinreina/Documents/PabloCatalan/REPOSITORY/data/nodes_data.csv')
df.columns = ['id', 'label', 'year', 'num_resistant', 'count', 'frequency']
df['year'] = df['year'].astype(int)

YEARS = sorted(df['year'].unique())
YEAR_MIN, YEAR_MAX = YEARS[0], YEARS[-1]
MIN_YEARS = 3   # minimum years a profile must appear in to be included in trend analysis

print(f"Years: {YEARS}")
print(f"Total nodes: {len(df)} | Unique profiles: {df['label'].nunique()}")

---
## A — Trend ranking: profiles increasing or decreasing over time

For each profile present in at least `MIN_YEARS` years, we fit a linear regression of frequency ~ year and use the slope as a trend metric.

In [ ]:
# Pivot: rows = profiles, columns = years
pivot = df.pivot_table(index='label', columns='year', values='frequency', aggfunc='first')

# Keep only profiles with enough years of data
pivot_filtered = pivot[pivot.notna().sum(axis=1) >= MIN_YEARS]
print(f"Profiles with {MIN_YEARS}+ years of data: {len(pivot_filtered)}")

# Also store num_resistant per profile for coloring
num_res = df.drop_duplicates('label').set_index('label')['num_resistant']

# Compute linear trend (slope) per profile
def compute_slope(row):
    valid = row.dropna()
    if len(valid) < 2:
        return np.nan
    x = np.array(valid.index.astype(int)) - YEAR_MIN
    return np.polyfit(x, valid.values, 1)[0]

slopes = pivot_filtered.apply(compute_slope, axis=1)
slopes = slopes.dropna().sort_values()

# Also compute absolute change (last available year - first available year)
def abs_change(row):
    valid = row.dropna()
    return valid.iloc[-1] - valid.iloc[0]

changes = pivot_filtered.apply(abs_change, axis=1).reindex(slopes.index)

trend_df = pd.DataFrame({
    'slope': slopes,
    'abs_change': changes,
    'num_resistant': num_res.reindex(slopes.index)
}).sort_values('slope')

TOP_N = 15  # how many profiles to show at each extreme

print(f"\nTop {TOP_N} INCREASING profiles (by slope):")
print(trend_df.tail(TOP_N)[['slope','abs_change','num_resistant']].iloc[::-1].to_string())
print(f"\nTop {TOP_N} DECREASING profiles (by slope):")
print(trend_df.head(TOP_N)[['slope','abs_change','num_resistant']].to_string())

In [ ]:
# --- Figure A1: slope ranking bar chart ---
top_inc = trend_df.tail(TOP_N).iloc[::-1]
top_dec = trend_df.head(TOP_N)

fig, axes = plt.subplots(1, 2, figsize=(16, 7))

for ax, data, title, color in zip(
    axes,
    [top_inc, top_dec],
    [f'Top {TOP_N} increasing profiles', f'Top {TOP_N} decreasing profiles'],
    ['#2a9d8f', '#e76f51']
):
    bars = ax.barh(range(len(data)), data['slope'] * 100, color=color, alpha=0.85)
    ax.set_yticks(range(len(data)))
    ax.set_yticklabels(data.index, fontsize=7.5)
    ax.set_xlabel('Slope (frequency % per year)', fontsize=9)
    ax.set_title(title, fontweight='bold')
    ax.axvline(0, color='black', linewidth=0.8)
    # Annotate num_resistant
    for i, (label, row) in enumerate(data.iterrows()):
        ax.text(
            data['slope'].max() * 100 * 0.02 if color == '#2a9d8f' else data['slope'].min() * 100 * 0.02,
            i, f"  n={int(row['num_resistant'])}", va='center', fontsize=7, color='gray'
        )

plt.suptitle('Temporal trend in relative frequency of resistance profiles', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# --- Figure A2: trajectory lines for top increasing and decreasing profiles ---
def plot_trajectories(profiles, title, color, ax):
    cmap = plt.get_cmap('YlOrRd') if color == 'red' else plt.get_cmap('YlGnBu')
    for i, label in enumerate(profiles):
        row = pivot.loc[label].dropna()
        c = cmap(0.3 + 0.7 * i / len(profiles))
        ax.plot(row.index.astype(int), row.values * 100, marker='o', markersize=4,
                color=c, linewidth=1.5, label=label)
    ax.set_title(title, fontweight='bold')
    ax.set_xlabel('Year')
    ax.set_ylabel('Relative frequency (%)')
    ax.set_xticks(YEARS)
    ax.legend(fontsize=6.5, loc='upper left', framealpha=0.5)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
plot_trajectories(top_inc.index[:10], 'Top 10 increasing', 'blue', axes[0])
plot_trajectories(top_dec.index[:10], 'Top 10 decreasing', 'red', axes[1])

plt.suptitle('Frequency trajectories over time', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

---
## B — Evolution by resistance burden zone

We group profiles by `num_resistant` and track how the total frequency of each group evolves per year.
This shows whether the population is shifting toward more or less resistance overall.

### Some figures

In [ ]:
# Total frequency per (year, num_resistant)
group = df.groupby(['year', 'num_resistant'])['frequency'].sum().reset_index()
group_pivot = group.pivot(index='year', columns='num_resistant', values='frequency').fillna(0)

print("Frequency by resistance group per year:")
print((group_pivot * 100).round(2).to_string())

In [ ]:
# --- Figure B1: stacked area chart ---
# Group into broader zones for readability
zones = {
    'Susceptible (0)':     [0],
    'Low (1-3)':           [1, 2, 3],
    'Moderate (4-6)':      [4, 5, 6],
    'High (7-10)':         [7, 8, 9, 10],
    'Very high (11+)':     list(range(11, 20))
}
zone_colors = ['#264653', '#2a9d8f', '#e9c46a', '#f4a261', '#e76f51']

zone_df = pd.DataFrame(index=YEARS)
for zone_name, nums in zones.items():
    cols = [c for c in group_pivot.columns if c in nums]
    zone_df[zone_name] = group_pivot[cols].sum(axis=1) * 100

fig, ax = plt.subplots(figsize=(11, 6))
zone_df.plot.area(ax=ax, color=zone_colors, alpha=0.85, linewidth=0)
ax.set_xlabel('Year')
ax.set_ylabel('Population frequency (%)')
ax.set_title('Distribution of resistance burden over time', fontweight='bold')
ax.set_xticks(YEARS)
ax.legend(loc='upper right', fontsize=9)
ax.set_ylim(0, 100)
plt.tight_layout()
plt.show()

In [ ]:
# --- Figure B2: line plot per num_resistant (fine-grained) ---
max_res = int(df['num_resistant'].max())
cmap = cm.get_cmap('RdYlGn_r', max_res + 1)

fig, ax = plt.subplots(figsize=(12, 6))
for n in range(max_res + 1):
    if n in group_pivot.columns:
        vals = group_pivot[n] * 100
        ax.plot(YEARS, vals, marker='o', markersize=4,
                color=cmap(n), linewidth=1.8, label=f'{n} resistances')

ax.set_xlabel('Year')
ax.set_ylabel('Total relative frequency (%)')
ax.set_title('Frequency per resistance count over time', fontweight='bold')
ax.set_xticks(YEARS)
ax.legend(fontsize=7, ncol=2, loc='upper right', framealpha=0.5)
plt.tight_layout()
plt.show()

In [ ]:
# --- Figure B3: heatmap (year x num_resistant) ---
fig, ax = plt.subplots(figsize=(14, 5))
im = ax.imshow(
    (group_pivot * 100).T,
    aspect='auto', cmap='YlOrRd',
    extent=[YEAR_MIN - 0.5, YEAR_MAX + 0.5,
            group_pivot.columns.max() + 0.5, group_pivot.columns.min() - 0.5]
)
ax.set_xlabel('Year')
ax.set_ylabel('Number of resistances')
ax.set_title('Heatmap: relative frequency by resistance burden and year', fontweight='bold')
ax.set_xticks(YEARS)
ax.set_yticks(range(0, max_res + 1))
plt.colorbar(im, ax=ax, label='Relative frequency (%)')
plt.tight_layout()
plt.show()

### Figure showing polarisation of resistance groups

In [ ]:
# Define 4 clinically meaningful groups
def assign_group(n):
    if n == 0:    return 'Susceptible\n(0)'
    elif n <= 3:  return 'Low resistance\n(1–3)'
    elif n <= 8:  return 'Moderate resistance\n(4–8)'
    elif n <= 12: return 'High resistance\n(9–12)'
    else:         return 'Extreme resistance\n(≥13)'

group_order  = ['Susceptible\n(0)', 'Low resistance\n(1–3)',
                'Moderate resistance\n(4–8)', 'High resistance\n(9–12)',
                'Extreme resistance\n(≥13)']
group_colors = ['#1b4332', '#40916c', '#95d5b2', '#f4a261', '#e76f51']

df['group'] = df['num_resistant'].apply(assign_group)
group_year = df.groupby(['year', 'group'])['frequency'].sum().unstack(fill_value=0) * 100
group_year = group_year[group_order]

fig, ax = plt.subplots(figsize=(12, 6))
x = np.arange(len(YEARS))
n_groups = len(group_order)
width = 0.15
offsets = np.linspace(-(n_groups-1)/2, (n_groups-1)/2, n_groups) * width

for i, (grp, col) in enumerate(zip(group_order, group_colors)):
    vals = group_year[grp].values
    ax.bar(x + offsets[i], vals, width, label=grp, color=col, alpha=0.9)
    slope = np.polyfit(range(len(vals)), vals, 1)[0]
    if abs(slope) > 0.05:
        arrow = '▲' if slope > 0 else '▼'
        ax.text(x[-1] + offsets[i] + 0.01, vals[-1] + 0.3,
                arrow, fontsize=8, color=col, ha='center', fontweight='bold')

ax.set_xticks(x)
ax.set_xticklabels(YEARS)
ax.set_xlabel('Year', fontsize=11)
ax.set_ylabel('Population frequency (%)', fontsize=11)
ax.set_title('Temporal evolution of resistance burden in E. coli\n(ATLAS 2018–2024)',
             fontsize=12, fontweight='bold')
ax.legend(title='Resistance group', fontsize=5.5, title_fontsize=5.5,
          loc='upper left', framealpha=0.8)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

plt.tight_layout()
plt.show()

### Polarisation by country (top 4 by number of isolates)

In [ ]:
# Load original ATLAS data
atlas = pd.read_csv('/Users/jimenamartinreina/Documents/PabloCatalan/DATABASES/VivliDataChallenge/Vivli_Escherichia_coli.csv')
# exclude colistin because you then define Susceptible node as resistant to 0 antibiotics

#resistance_columns = [col for col in atlas.columns if col.endswith('_I') and col != 'Colistin_I']
resistance_columns = ["Ciprofloxacin_I", "Levofloxacin_I", "Meropenem_I", "Imipenem_I", "Ceftazidime_I",
                      "Cefepime_I", "Ceftaroline_I", "Ampicillin_I", "Ampicillin sulbactam_I", 
                      "Amoxycillin clavulanate_I", "Piperacillin tazobactam_I", "Aztreonam_I", "Amikacin_I", 
                      "Gentamicin_I", "Trimethoprim sulfa_I"] # just 15 because of claude model

# Binarize
atlas_bin = atlas[['Year', 'Country'] + resistance_columns].copy()
for col in resistance_columns:
    atlas_bin[col] = (atlas_bin[col] == 'Resistant').astype(int)
atlas_bin['num_resistant'] = atlas_bin[resistance_columns].sum(axis=1)

# Top 4 countries by number of isolates
top_countries = atlas_bin['Country'].value_counts().head(4).index.tolist()
print("Top 4 countries:", top_countries)

# Same group definition as Figure B
def assign_group(n):
    if n == 0:    return 'Susceptible\n(0)'
    elif n <= 3:  return 'Low resistance\n(1–3)'
    elif n <= 8:  return 'Moderate resistance\n(4–8)'
    elif n <= 12: return 'High resistance\n(9–12)'
    else:         return 'Extreme resistance\n(≥13)'

group_order  = ['Susceptible\n(0)', 'Low resistance\n(1–3)',
                'Moderate resistance\n(4–8)', 'High resistance\n(9–12)',
                'Extreme resistance\n(≥13)']
group_colors = ['#1b4332', '#40916c', '#95d5b2', '#f4a261', '#e76f51']

atlas_bin['group'] = atlas_bin['num_resistant'].apply(assign_group)

# --- Figure D: one subplot per country ---
fig, axes = plt.subplots(2, 2, figsize=(13, 10), sharey=True)
axes = axes.flatten()

for ax, country in zip(axes, top_countries):
    country_df = atlas_bin[atlas_bin['Country'] == country]
    years = sorted(country_df['Year'].unique())

    # Relative frequency per group per year
    group_year = (country_df.groupby(['Year', 'group'])
                  .size()
                  .unstack(fill_value=0))
    group_year = group_year.div(group_year.sum(axis=1), axis=0) * 100
    # Ensure all groups are present
    for g in group_order:
        if g not in group_year.columns:
            group_year[g] = 0
    group_year = group_year[group_order]

    x = np.arange(len(years))
    n_groups = len(group_order)
    width = 0.15
    offsets = np.linspace(-(n_groups-1)/2, (n_groups-1)/2, n_groups) * width

    for i, (grp, col) in enumerate(zip(group_order, group_colors)):
        vals = group_year[grp].values
        ax.bar(x + offsets[i], vals, width, label=grp, color=col, alpha=0.9)
        if len(vals) >= 2:
            slope = np.polyfit(range(len(vals)), vals, 1)[0]
            if abs(slope) > 0.05:
                arrow = '▲' if slope > 0 else '▼'
                ax.text(x[-1] + offsets[i], vals[-1] + 0.5,
                        arrow, fontsize=8, color=col, ha='center', fontweight='bold')

    ax.set_title(country, fontweight='bold', fontsize=15)
    ax.set_xticks(x)
    ax.set_xticklabels(years, fontsize=15)
    ax.set_xlabel('Year', fontsize=15)
    ax.set_ylabel('Population frequency (%)', fontsize=15)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    n_isolates = len(country_df)
    ax.text(0.02, 1.0, f'n = {n_isolates:,} isolates',
            transform=ax.transAxes, fontsize=15, color='gray', va='top')

# Shared legend
handles = [plt.Rectangle((0,0),1,1, color=c, alpha=0.9) for c in group_colors]
fig.legend(handles, [g.replace('\n', ' ') for g in group_order],
           title='Resistance group', fontsize=13, title_fontsize=15,
           loc='lower center', ncol=5, bbox_to_anchor=(0.5, -0.02), framealpha=0.8)

# plt.suptitle('Polarisation of resistance burden by country (top 4)\n(ATLAS 2018–2024)',
#              fontsize=25, fontweight='bold')
plt.tight_layout(rect=[0, 0.05, 1, 1])

plt.savefig(f'/Users/jimenamartinreina/Documents/PabloCatalan/REPOSITORY/outputs/figures/resistance_tendencies_countries.png', dpi=100, bbox_inches='tight')
plt.show()


---
## C — Direct comparison: first year vs last year (2018 vs 2024)

For profiles present in both years, we compute the absolute and relative change in frequency.
We also flag profiles that emerged (only in 2024) or disappeared (only in 2018).

In [ ]:
df_first = df[df['year'] == YEAR_MIN].set_index('label')[['frequency', 'num_resistant']]
df_last  = df[df['year'] == YEAR_MAX].set_index('label')[['frequency', 'num_resistant']]

# Profiles in both years
common = df_first.index.intersection(df_last.index)
comparison = pd.DataFrame({
    f'freq_{YEAR_MIN}': df_first.loc[common, 'frequency'],
    f'freq_{YEAR_MAX}': df_last.loc[common, 'frequency'],
    'num_resistant': df_first.loc[common, 'num_resistant']
})
comparison['abs_change'] = comparison[f'freq_{YEAR_MAX}'] - comparison[f'freq_{YEAR_MIN}']
comparison['rel_change'] = comparison['abs_change'] / comparison[f'freq_{YEAR_MIN}'] * 100
comparison = comparison.sort_values('abs_change')

# Emerged and disappeared
emerged     = df_last.index.difference(df_first.index)
disappeared = df_first.index.difference(df_last.index)

print(f"Profiles in both {YEAR_MIN} and {YEAR_MAX}: {len(common)}")
print(f"Emerged in {YEAR_MAX} (not in {YEAR_MIN}): {len(emerged)}")
print(f"Disappeared by {YEAR_MAX} (not in {YEAR_MAX}): {len(disappeared)}")

print(f"\nTop 10 most increased (absolute):")
print(comparison.tail(10)[['abs_change','rel_change','num_resistant']].iloc[::-1].round(5).to_string())
print(f"\nTop 10 most decreased (absolute):")
print(comparison.head(10)[['abs_change','rel_change','num_resistant']].round(5).to_string())

In [ ]:
# --- Figure C1: scatter 2018 vs 2024 frequency, colored by change ---
fig, ax = plt.subplots(figsize=(9, 8))

vmax = comparison['abs_change'].abs().quantile(0.98)
norm = mcolors.TwoSlopeNorm(vmin=-vmax, vcenter=0, vmax=vmax)
cmap_div = cm.get_cmap('RdYlGn')

sc = ax.scatter(
    comparison[f'freq_{YEAR_MIN}'] * 100,
    comparison[f'freq_{YEAR_MAX}'] * 100,
    c=comparison['abs_change'],
    cmap=cmap_div, norm=norm,
    s=comparison['num_resistant'] * 8 + 15,
    alpha=0.75, edgecolors='none'
)

# Diagonal = no change
lim = max(comparison[f'freq_{YEAR_MIN}'].max(), comparison[f'freq_{YEAR_MAX}'].max()) * 100 * 1.05
ax.plot([0, lim], [0, lim], '--', color='gray', linewidth=1, label='No change')

# Label top movers
TOP_LABEL = 8
to_label = pd.concat([comparison.head(TOP_LABEL), comparison.tail(TOP_LABEL)])
for label, row in to_label.iterrows():
    ax.annotate(
        label, 
        xy=(row[f'freq_{YEAR_MIN}'] * 100, row[f'freq_{YEAR_MAX}'] * 100),
        fontsize=5.5, ha='left', va='bottom',
        xytext=(4, 2), textcoords='offset points',
        arrowprops=dict(arrowstyle='-', color='gray', lw=0.5)
    )

plt.colorbar(sc, ax=ax, label='Absolute change in frequency')
ax.set_xlabel(f'Relative frequency in {YEAR_MIN} (%)')
ax.set_ylabel(f'Relative frequency in {YEAR_MAX} (%)')
ax.set_title(f'{YEAR_MIN} vs {YEAR_MAX}: per-profile frequency change\n(dot size = num_resistant)', fontweight='bold')
ax.legend(fontsize=8)
plt.tight_layout()
plt.show()

In [ ]:
# --- Figure C2: bar chart of top movers (absolute change) ---
TOP_N_C = 20
top_movers = pd.concat([comparison.head(TOP_N_C), comparison.tail(TOP_N_C)]).drop_duplicates()
top_movers = top_movers.sort_values('abs_change')

colors = ['#e76f51' if v < 0 else '#2a9d8f' for v in top_movers['abs_change']]

fig, ax = plt.subplots(figsize=(12, 10))
ax.barh(range(len(top_movers)), top_movers['abs_change'] * 100, color=colors, alpha=0.85)
ax.set_yticks(range(len(top_movers)))
ax.set_yticklabels(top_movers.index, fontsize=7)
ax.axvline(0, color='black', linewidth=0.8)
ax.set_xlabel(f'Change in relative frequency (%) between {YEAR_MIN} and {YEAR_MAX}')
ax.set_title(f'Biggest movers: {YEAR_MIN} → {YEAR_MAX}', fontweight='bold')

# Annotate num_resistant
for i, (label, row) in enumerate(top_movers.iterrows()):
    ax.text(0.001, i, f" n={int(row['num_resistant'])}", va='center', fontsize=6.5, color='gray')

legend_elements = [
    Line2D([0], [0], color='#2a9d8f', lw=8, label='Increased'),
    Line2D([0], [0], color='#e76f51', lw=8, label='Decreased')
]
ax.legend(handles=legend_elements, fontsize=9)
plt.tight_layout()
plt.show()

In [ ]:
# --- Emerged and disappeared profiles ---
print(f"=== Profiles that EMERGED in {YEAR_MAX} (not present in {YEAR_MIN}) ===")
emerged_df = df_last.loc[emerged].sort_values('frequency', ascending=False)
print(emerged_df.head(20).to_string())

print(f"\n=== Profiles that DISAPPEARED by {YEAR_MAX} (not present in {YEAR_MAX}) ===")
disappeared_df = df_first.loc[disappeared].sort_values('frequency', ascending=False)
print(disappeared_df.head(20).to_string())

In [ ]:
# --- Figure C3: emerged vs disappeared by resistance burden ---
emerged_by_n     = df_last.loc[emerged, 'num_resistant'].value_counts().sort_index()
disappeared_by_n = df_first.loc[disappeared, 'num_resistant'].value_counts().sort_index()

all_n = sorted(set(emerged_by_n.index) | set(disappeared_by_n.index))
emerged_by_n     = emerged_by_n.reindex(all_n, fill_value=0)
disappeared_by_n = disappeared_by_n.reindex(all_n, fill_value=0)

fig, ax = plt.subplots(figsize=(12, 5))
x = np.array(all_n)
width = 0.4
ax.bar(x - width/2, emerged_by_n.values,     width, color='#2a9d8f', alpha=0.85, label=f'Emerged in {YEAR_MAX}')
ax.bar(x + width/2, disappeared_by_n.values, width, color='#e76f51', alpha=0.85, label=f'Disappeared by {YEAR_MAX}')
ax.set_xlabel('Number of resistances')
ax.set_ylabel('Number of profiles')
ax.set_title(f'Profiles emerged vs disappeared by resistance burden ({YEAR_MIN}→{YEAR_MAX})', fontweight='bold')
ax.set_xticks(all_n)
ax.legend(fontsize=9)
plt.tight_layout()
plt.show()

---
## Summary of key findings

Run this cell last to print a concise summary.

In [ ]:
print("="*60)
print("SUMMARY")
print("="*60)

print("\n[A] TREND (slope, profiles with 3+ years)")
print(f"  Most increasing: {trend_df.tail(1).index[0]}")
print(f"    slope = {trend_df['slope'].iloc[-1]*100:.4f}% per year")
print(f"  Most decreasing: {trend_df.head(1).index[0]}")
print(f"    slope = {trend_df['slope'].iloc[0]*100:.4f}% per year")

print("\n[B] ZONE SHIFTS")
for zone_name in zone_df.columns:
    delta = zone_df[zone_name].iloc[-1] - zone_df[zone_name].iloc[0]
    direction = '▲' if delta > 0 else '▼'
    print(f"  {zone_name}: {direction} {abs(delta):.2f}%")

print(f"\n[C] {YEAR_MIN} vs {YEAR_MAX}")
print(f"  Profiles in both years: {len(common)}")
print(f"  Emerged in {YEAR_MAX}: {len(emerged)}")
print(f"  Disappeared by {YEAR_MAX}: {len(disappeared)}")
print(f"  Biggest absolute increase: {comparison.tail(1).index[0]}")
print(f"    Δ = {comparison['abs_change'].iloc[-1]*100:+.4f}%")
print(f"  Biggest absolute decrease: {comparison.head(1).index[0]}")
print(f"    Δ = {comparison['abs_change'].iloc[0]*100:+.4f}%")

## Extra: calculate % of multiresistant isolates by each year

This is the code I wrote to study how the % of multiresistant isolates changed, before creating this network_temporal_analysis.ipynb

I use data from the ATLAS cleaned E. coli version

In [ ]:
df1=pd.read_csv('/Users/jimenamartinreina/Documents/PabloCatalan/DATABASES/VivliDataChallenge/Vivli_E.coli.csv')

In [ ]:
resistance_columns = [col for col in df1.columns if col.endswith('_I')]

In [ ]:
# calculate % of multiresistant isolates (>15 resistant antibiotics) in each year
# you have to sum when the value in resistance_columns is not S and then check if the sum is greater than 15
for i in range(0, 21):
    df1['multiresistant_' + str(i)] = df1[resistance_columns].apply(lambda x: sum(x == 'Resistant'), axis=1) > i

In [ ]:
# # in case you want the specific values of each %
# for i in range(15, 21):
#     multiresistant_percentage = df1.groupby('Year')['multiresistant_' + str(i)].mean() * 100
#     print("Percentage of multiresistant isolates (>" + str(i) + " resistant antibiotics) by year:")
#     print(multiresistant_percentage)

In [ ]:
# plot all the multiresistant_percentage in the same graph with a different color for each line
plt.figure(figsize=(12,8))
for i in range(0, 21):
    multiresistant_percentage = df1.groupby('Year')['multiresistant_' + str(i)].mean() * 100
    plt.plot(multiresistant_percentage.index, multiresistant_percentage.values, label='>' + str(i) + ' resistant antibiotics')
plt.title('Percentage of Multiresistant Isolates by Year')
plt.xlabel('Year')
plt.ylabel('Percentage of Multiresistant Isolates')
plt.xticks(rotation=45)
plt.legend()
plt.tight_layout()
plt.show()